# 04. CPU vs GPU 성능 비교 — 보안 탐지 모델 (WAF · IPS · DDoS)

**목적**: 1~3단계에서 만든 탐지 모델을 같은 데이터·같은 하이퍼파라미터로 CPU와 GPU에서 학습·추론하여
**학습 시간 / 추론 처리량 / 지연 시간**을 비교하고, 실서비스 트래픽 기준 **용량 산정(capacity planning)** 결론을 낸다.

**실행 환경**: Google Colab → 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`
(GPU가 없으면 CPU 측정만 수행하고 GPU 항목은 건너뜀)

| 모델 | CPU | GPU |
|---|---|---|
| XGBoost (트리 기반) | `device="cpu"`, `tree_method="hist"` | `device="cuda"` |
| MLP (PyTorch) | `torch.device("cpu")` | `torch.device("cuda")` + FP16(AMP) |


## 1. 환경 정보 수집
측정 결과는 하드웨어에 따라 달라지므로 환경 정보를 결과와 함께 저장한다.

In [ ]:
import os, json, platform, subprocess, time
import numpy as np, pandas as pd

RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        return f"N/A ({e})"

import torch, xgboost, sklearn
HAS_GPU = torch.cuda.is_available()

env = {
    "python": platform.python_version(),
    "cpu_model": sh("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'"),
    "cpu_cores": os.cpu_count(),
    "ram_gb": round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3, 1),
    "gpu": torch.cuda.get_device_name(0) if HAS_GPU else None,
    "gpu_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if HAS_GPU else None,
    "nvidia_driver": sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader") if HAS_GPU else None,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if HAS_GPU else None,
    "xgboost": xgboost.__version__,
    "sklearn": sklearn.__version__,
}
json.dump(env, open(f"{RESULT_DIR}/env.json", "w"), indent=2, ensure_ascii=False)
pd.Series(env)

In [ ]:
!nvidia-smi || echo "GPU 없음 — CPU 측정만 진행합니다" 

## 2. 데이터 설정
1~3단계에서 저장한 **특징(feature) CSV**가 있으면 경로를 지정한다. (숫자형 컬럼 + 라벨 컬럼)
경로가 없거나 파일이 없으면 각 단계와 비슷한 크기의 **합성 데이터**로 대체한다 — 성능 비교가 목적이므로 정확도보다 *데이터 크기와 형태*가 중요하다.

In [ ]:
DATA_DIR = "/content/data"          # 1~3단계 특징 CSV를 업로드한 폴더
LABEL_COL = "label"

STAGES = {
    # 단계: (CSV 파일명, 합성 시 샘플 수, 특징 수, 클래스 수)
    "WAF":  ("waf_features.csv",  200_000,  64, 2),
    "IPS":  ("ips_features.csv",  300_000,  80, 5),
    "DDoS": ("ddos_features.csv", 500_000,  40, 2),
}

QUICK = False   # True: 샘플 수를 1/10로 줄여 빠르게 동작 확인

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def load_stage(name):
    fname, n, f, k = STAGES[name]
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        y = pd.factorize(df[LABEL_COL])[0]
        X = df.drop(columns=[LABEL_COL]).select_dtypes("number").fillna(0).to_numpy(np.float32)
        src = "real"
    else:
        n = n // 10 if QUICK else n
        X, y = make_classification(n_samples=n, n_features=f, n_informative=f // 2,
                                   n_classes=k, weights=None if k > 2 else [0.9, 0.1],
                                   random_state=42)
        X = X.astype(np.float32)
        src = "synthetic"
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"[{name}] {src}: train {Xtr.shape}, test {Xte.shape}, classes {len(np.unique(y))}")
    return Xtr, Xte, ytr, yte, src

DATA = {s: load_stage(s) for s in STAGES}
SRC = {s: DATA[s][4] for s in DATA}
if "synthetic" in SRC.values():
    print("\n⚠️  합성 데이터 사용 단계:", [s for s, v in SRC.items() if v == "synthetic"],
          "\n   → 이 단계의 F1은 '탐지 성능'이 아니라 CPU/GPU 결과 일치 확인용입니다. 속도 비교로만 해석하세요.")

## 3. 측정 도구
GPU 연산은 **비동기**라서 그냥 시간을 재면 실제보다 짧게 나온다 → 측정 전후에 `torch.cuda.synchronize()`.
첫 실행은 CUDA 컨텍스트 초기화·커널 로딩 때문에 느리므로 **워밍업 후** 여러 번 반복해 중앙값을 쓴다.

In [ ]:
def sync():
    if HAS_GPU:
        torch.cuda.synchronize()

def bench(fn, repeat=5, warmup=1):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeat):
        sync(); t = time.perf_counter()
        fn()
        sync(); times.append(time.perf_counter() - t)
    return float(np.median(times))

def latency(fn, n=200, warmup=20):
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(n):
        sync(); t = time.perf_counter()
        fn()
        sync(); ts.append((time.perf_counter() - t) * 1000)
    return float(np.percentile(ts, 50)), float(np.percentile(ts, 99))

DEVICES = ["cpu"] + (["cuda"] if HAS_GPU else [])
BATCHES = [1, 64, 1024, 16384]
ROWS = []   # 모든 측정 결과를 여기에 모은다

## 4. XGBoost — 학습 시간 · 정확도

In [ ]:
import xgboost as xgb
from sklearn.metrics import f1_score
try:
    import cupy as cp          # Colab 기본 설치 — GPU 모델에는 GPU 배열을 넘겨 장치 불일치 경고·복사 방지
except ImportError:
    cp = None

def to_dev(X, dev):
    return cp.asarray(X) if (dev == "cuda" and cp is not None) else X

def xgb_predict(booster, X, dev):
    p = booster.inplace_predict(to_dev(X, dev))
    p = cp.asnumpy(p) if (cp is not None and isinstance(p, cp.ndarray)) else np.asarray(p)
    return p.argmax(1) if p.ndim == 2 else (p > 0.5).astype(int)

XGB_PARAMS = dict(n_estimators=300, max_depth=8, learning_rate=0.1,
                  tree_method="hist", n_jobs=-1, random_state=42)
XGB_MODELS = {}

for stage, (Xtr, Xte, ytr, yte, src) in DATA.items():
    for dev in DEVICES:
        model = xgb.XGBClassifier(device=dev, **XGB_PARAMS)
        sync(); t = time.perf_counter()
        model.fit(Xtr, ytr)
        sync(); train_s = time.perf_counter() - t
        f1 = f1_score(yte, xgb_predict(model.get_booster(), Xte, dev), average="macro")
        XGB_MODELS[(stage, dev)] = model
        ROWS.append(dict(stage=stage, model="XGBoost", device=dev, metric="train_s", value=train_s))
        ROWS.append(dict(stage=stage, model="XGBoost", device=dev, metric="f1", value=f1))
        print(f"[{stage}] XGBoost {dev:4s}  학습 {train_s:7.2f}s  F1 {f1:.4f}")

## 5. XGBoost — 추론 처리량 · 지연 시간
- **처리량**: 배치 크기별 초당 처리 건수 (rows/s)
- **지연 시간**: 요청 1건(batch=1)의 p50 / p99 (ms) — 실시간 차단(WAF 인라인)에서 중요한 지표

GPU 추론은 입력도 GPU 메모리에 있어야 공정하다 → `cupy`가 있으면 GPU 배열로 넘기고(`to_dev`), 없으면 numpy 입력(전송 비용 포함)으로 측정한다.

In [ ]:
for stage, (Xtr, Xte, ytr, yte, src) in DATA.items():
    for dev in DEVICES:
        booster = XGB_MODELS[(stage, dev)].get_booster()
        for b in BATCHES:
            xb = Xte[:b] if b <= len(Xte) else np.resize(Xte, (b, Xte.shape[1]))
            xb = to_dev(xb, dev)
            run = lambda: booster.inplace_predict(xb)
            sec = bench(run, repeat=5 if b > 1 else 50)
            ROWS.append(dict(stage=stage, model="XGBoost", device=dev, metric=f"rows_per_s@{b}", value=b / sec))
        x1 = to_dev(Xte[:1], dev)
        p50, p99 = latency(lambda: booster.inplace_predict(x1))
        ROWS.append(dict(stage=stage, model="XGBoost", device=dev, metric="lat_p50_ms", value=p50))
        ROWS.append(dict(stage=stage, model="XGBoost", device=dev, metric="lat_p99_ms", value=p99))
        print(f"[{stage}] XGBoost {dev:4s}  batch=1 p50 {p50:.3f}ms p99 {p99:.3f}ms")

## 6. MLP (PyTorch) — 학습 시간 · 추론 처리량

In [ ]:
import torch.nn as nn

EPOCHS, BS = 5, 1024

def make_mlp(n_in, n_out):
    return nn.Sequential(nn.Linear(n_in, 256), nn.ReLU(), nn.Dropout(0.1),
                         nn.Linear(256, 128), nn.ReLU(),
                         nn.Linear(128, n_out))

def train_mlp(Xtr, ytr, dev, amp=False):
    torch.manual_seed(42)
    d = torch.device(dev)
    X = torch.from_numpy(Xtr).to(d); y = torch.from_numpy(ytr.astype(np.int64)).to(d)
    model = make_mlp(X.shape[1], int(y.max()) + 1).to(d)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    sync(); t = time.perf_counter()
    for _ in range(EPOCHS):
        perm = torch.randperm(len(X), device=d)
        for i in range(0, len(X), BS):
            idx = perm[i:i + BS]
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                loss = loss_fn(model(X[idx]), y[idx])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sync()
    return model.eval(), time.perf_counter() - t

MLP_VARIANTS = [("cpu", False)] + ([("cuda", False), ("cuda", True)] if HAS_GPU else [])

for stage, (Xtr, Xte, ytr, yte, src) in DATA.items():
    for dev, amp in MLP_VARIANTS:
        label = dev + ("+fp16" if amp else "")
        model, train_s = train_mlp(Xtr, ytr, dev, amp)
        d = torch.device(dev)
        Xt = torch.from_numpy(Xte).to(d)
        with torch.inference_mode(), torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
            pred = model(Xt).argmax(1).cpu().numpy()
            f1 = f1_score(yte, pred, average="macro")
            for b in BATCHES:
                xb = Xt[:b] if b <= len(Xt) else Xt.repeat(b // len(Xt) + 1, 1)[:b]
                sec = bench(lambda: model(xb), repeat=5 if b > 1 else 50)
                ROWS.append(dict(stage=stage, model="MLP", device=label, metric=f"rows_per_s@{b}", value=b / sec))
            p50, p99 = latency(lambda: model(Xt[:1]))
        for m, v in [("train_s", train_s), ("f1", f1), ("lat_p50_ms", p50), ("lat_p99_ms", p99)]:
            ROWS.append(dict(stage=stage, model="MLP", device=label, metric=m, value=v))
        print(f"[{stage}] MLP {label:10s} 학습 {train_s:7.2f}s  F1 {f1:.4f}  p50 {p50:.3f}ms")

## 7. 결과 정리

In [ ]:
res = pd.DataFrame(ROWS)
res.to_csv(f"{RESULT_DIR}/benchmark_raw.csv", index=False)

table = res.pivot_table(index=["stage", "model", "device"], columns="metric", values="value")
cols = ["train_s", "f1", "lat_p50_ms", "lat_p99_ms"] + [f"rows_per_s@{b}" for b in BATCHES]
table = table[[c for c in cols if c in table.columns]]
table.to_csv(f"{RESULT_DIR}/benchmark_summary.csv")
table.style.format("{:,.3f}")

In [ ]:
# GPU 가속 배율 (CPU 대비) — 학습 시간은 작을수록, 처리량은 클수록 좋음
if HAS_GPU:
    sp = []
    for (stage, model), g in table.groupby(level=[0, 1]):
        cpu = g.xs("cpu", level=2).iloc[0]
        for dev in g.index.get_level_values(2):
            if dev == "cpu":
                continue
            r = g.xs(dev, level=2).iloc[0]
            row = {"stage": stage, "model": model, "device": dev, "train": cpu["train_s"] / r["train_s"]}
            for b in BATCHES:
                row[f"thr@{b}"] = r[f"rows_per_s@{b}"] / cpu[f"rows_per_s@{b}"]
            sp.append(row)
    speedup = pd.DataFrame(sp).set_index(["stage", "model", "device"])
    speedup.to_csv(f"{RESULT_DIR}/speedup.csv")
    display(speedup.style.format("{:.2f}x").background_gradient(cmap="RdYlGn", vmin=0, vmax=5))

## 8. 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(STAGES), figsize=(5 * len(STAGES), 4), sharey=False)
for ax, stage in zip(np.atleast_1d(axes), STAGES):
    sub = res[(res.stage == stage) & res.metric.str.startswith("rows_per_s@")].copy()
    sub["batch"] = sub.metric.str.split("@").str[1].astype(int)
    for (model, dev), g in sub.groupby(["model", "device"]):
        g = g.sort_values("batch")
        ax.plot(g.batch, g.value, marker="o", label=f"{model} / {dev}")
    ax.set(xscale="log", yscale="log", title=f"{stage} inference throughput", xlabel="batch size", ylabel="rows/s")
    ax.grid(True, which="both", alpha=.3)
np.atleast_1d(axes)[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULT_DIR}/throughput_by_batch.png", dpi=150); plt.show()

tr = res[res.metric == "train_s"].pivot_table(index=["stage", "model"], columns="device", values="value")
ax = tr.plot.bar(figsize=(9, 4), title="Training time (s, lower is better)", rot=0)
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULT_DIR}/train_time.png", dpi=150); plt.show()

## 9. 용량 산정 (Capacity Planning)
측정한 처리량으로 **목표 트래픽을 처리하는 데 필요한 자원**을 계산한다.

- Colab CPU는 vCPU 2개뿐이라 서버 CPU보다 훨씬 불리하다 → CPU는 **코어당 처리량**으로 환산한 뒤 서버 코어 수로 다시 곱한다.
  이는 **선형 확장 가정(낙관치)** 이다. 실제로는 메모리 대역폭·스레드 경합 때문에 이보다 낮으므로 `CPU_SCALING_EFF`로 보정한다.
- 결과는 **필요 대수(최소 1대)** 와 **1대 기준 예상 사용률(%)** 로 표시한다.
- 운영 여유율(headroom)을 두어 평균 사용률 70% 이하로 설계한다.
- 비용은 사용 중인 클라우드·장비 단가를 직접 입력한다.

In [ ]:
# ===== 운영 조건 입력 =====
TARGET_RPS    = {"WAF": 5_000, "IPS": 50_000, "DDoS": 200_000}  # 단계별 목표 요청/플로우 수 (건/초)
BATCH_FOR_PLAN = 1024        # 실서비스 마이크로배치 크기 (지연 허용 범위 안에서)
HEADROOM       = 0.7         # 최대 사용률
SERVER_CORES   = 32          # 비교 대상 CPU 서버 코어 수
CPU_SCALING_EFF = 0.7        # 코어 수 확장 효율 (1.0 = 완전 선형, 보수적으로 0.6~0.8 권장)
COST_CPU_HR    = None        # CPU 서버 시간당 비용 (예: 1.5) — 모르면 None
COST_GPU_HR    = None        # GPU 서버 시간당 비용
# ==========================

col = f"rows_per_s@{BATCH_FOR_PLAN}"
plan = []
for (stage, model, dev), r in table.iterrows():
    if dev == "cpu":
        per_unit = r[col] / env["cpu_cores"] * SERVER_CORES * CPU_SCALING_EFF
        unit_name, cost, basis = f"{SERVER_CORES}코어 서버", COST_CPU_HR, f"환산(효율 {CPU_SCALING_EFF:.0%})"
    else:
        per_unit = r[col]
        unit_name, cost, basis = f"GPU {env['gpu']}", COST_GPU_HR, "실측"
    units = max(1, int(np.ceil(TARGET_RPS[stage] / (per_unit * HEADROOM))))
    util = TARGET_RPS[stage] / (per_unit * units) * 100
    plan.append({"stage": stage, "model": model, "device": dev,
                 "목표(건/s)": TARGET_RPS[stage],
                 "1대 처리량(건/s)": round(per_unit),
                 "근거": basis,
                 "필요 대수": units, "단위": unit_name,
                 "예상 사용률(%)": round(util, 2),
                 "시간당 비용": None if cost is None else round(units * cost, 2)})
plan = pd.DataFrame(plan).set_index(["stage", "model", "device"])
plan.to_csv(f"{RESULT_DIR}/capacity_plan.csv", encoding="utf-8-sig")
plan.style.format({"1대 처리량(건/s)": "{:,.0f}", "목표(건/s)": "{:,.0f}", "예상 사용률(%)": "{:.2f}"})

## 10. 결론 자동 요약 + 결과 다운로드

In [ ]:
lines = [f"# 결과 요약 ({env['gpu'] or 'GPU 없음'} / CPU {env['cpu_cores']}코어)", "",
         "## 전제", 
         "- 데이터: " + ", ".join(f"{s}={v}" for s, v in SRC.items()),
         "  - synthetic 단계의 F1은 탐지 성능이 아니라 CPU/GPU 결과 일치 확인용 (속도 비교 전용)",
         f"- CPU 측정은 Colab vCPU {env['cpu_cores']}개 기준. 서버 환산은 코어당 처리량 × {SERVER_CORES}코어 × 효율 {CPU_SCALING_EFF:.0%} (추정치)",
         "- 측정 범위는 모델 연산만 (로그 파싱·특징 추출 시간 제외)", "",
         "## 단계·모델별 최적 장치"]
for (stage, model), g in table.groupby(level=[0, 1]):
    devs = g.index.get_level_values(2)
    best_train = devs[g["train_s"].values.argmin()]
    best_lat = devs[g["lat_p50_ms"].values.argmin()]
    best_thr = devs[g[f"rows_per_s@{BATCHES[-1]}"].values.argmax()]
    lines.append(f"- **{stage} / {model}**: 학습 최단 `{best_train}`, "
                 f"단건 지연 최저 `{best_lat}`, 대량 배치 처리량 최고 `{best_thr}`")
lines += ["", "> 단건(batch=1) 요청은 GPU 전송·커널 실행 오버헤드 때문에 CPU가 유리할 수 있고, "
          "배치가 커질수록 GPU 이점이 커진다. 인라인 차단(WAF)은 지연, 로그·플로우 분석(IPS·DDoS)은 처리량 기준으로 판단한다.",
          "", f"## 용량 산정 (batch={BATCH_FOR_PLAN}, 최대 사용률 {HEADROOM:.0%})"]
for idx, r in plan.iterrows():
    lines.append(f"- {' / '.join(idx)}: {r['단위']} {r['필요 대수']}대, 예상 사용률 {r['예상 사용률(%)']}% ({r['근거']})")
summary = "\n".join(lines)
open(f"{RESULT_DIR}/SUMMARY.md", "w").write(summary)
print(summary)

In [ ]:
import shutil
shutil.make_archive("gpu_benchmark_results", "zip", RESULT_DIR)
try:
    from google.colab import files
    files.download("gpu_benchmark_results.zip")
except ImportError:
    print("Colab이 아니므로 gpu_benchmark_results.zip 파일을 직접 확인하세요.")